IMPORT IMPORTANT LIBRARIES

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

LOAD THE DATASET

In [3]:
df = pd.read_csv("/content/drive/MyDrive/DatasetAuspify.csv")
print("Dataset Loaded Successfully")

Dataset Loaded Successfully


In [4]:
df.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [5]:
df.shape

(8790, 10)

ANALYZE RATING CATEGORIES

In [6]:
# 1. Display unique rating categories
print("Unique Rating Categories:")
print(df["rating"].unique())

# 2. Count records in each rating category
print("\nRating Category Distribution:")
print(df["rating"].value_counts())

# 3. Check the number of unique categories
print("\nTotal Rating Categories:")
print(df["rating"].nunique())

# 4. Check missing values
print("\nMissing Values:")
print(df["rating"].isnull().sum())

# 5. Calculate rating percentages
print("\nRating Percentages:")
print((df["rating"].value_counts(normalize=True) * 100).round(2))

Unique Rating Categories:
['PG-13' 'TV-MA' 'TV-PG' 'TV-14' 'TV-Y7' 'TV-Y' 'PG' 'TV-G' 'R' 'G'
 'NC-17' 'NR' 'TV-Y7-FV' 'UR']

Rating Category Distribution:
rating
TV-MA       3205
TV-14       2157
TV-PG        861
R            799
PG-13        490
TV-Y7        333
TV-Y         306
PG           287
TV-G         220
NR            79
G             41
TV-Y7-FV       6
NC-17          3
UR             3
Name: count, dtype: int64

Total Rating Categories:
14

Missing Values:
0

Rating Percentages:
rating
TV-MA       36.46
TV-14       24.54
TV-PG        9.80
R            9.09
PG-13        5.57
TV-Y7        3.79
TV-Y         3.48
PG           3.27
TV-G         2.50
NR           0.90
G            0.47
TV-Y7-FV     0.07
NC-17        0.03
UR           0.03
Name: proportion, dtype: float64


PREPARE TRAINING DATASETS

In [7]:
# 1. Select input features
features = [
    "type",
    "release_year",
    "country",
    "listed_in",
    "director"
]

# 2. Create input and target variables
X = df[features].copy()
y = df["rating"].copy()

# 3. Display dataset shapes
print("Input Features Shape:", X.shape)
print("Target Shape:", y.shape)

# 4. Display selected features
display(X.head())

# 5. Display target distribution
print("\nTarget Distribution:")
print(y.value_counts())

# 6. Check missing values
print("\nMissing Values:")
print(X.isnull().sum())

Input Features Shape: (8790, 5)
Target Shape: (8790,)


,type,release_year,country,listed_in,director
0,Movie,2020,United States,Documentaries,Kirsten Johnson
1,TV Show,2021,France,"Crime TV Shows, International TV Shows, TV Act...",Julien Leclercq
2,TV Show,2021,United States,"TV Dramas, TV Horror, TV Mysteries",Mike Flanagan
3,Movie,2021,Brazil,"Children & Family Movies, Comedies",Bruno Garotti
4,Movie,1993,United States,"Dramas, Independent Movies, International Movies",Haile Gerima



Target Distribution:
rating
TV-MA       3205
TV-14       2157
TV-PG        861
R            799
PG-13        490
TV-Y7        333
TV-Y         306
PG           287
TV-G         220
NR            79
G             41
TV-Y7-FV       6
NC-17          3
UR             3
Name: count, dtype: int64

Missing Values:
type            0
release_year    0
country         0
listed_in       0
director        0
dtype: int64


TRAIN CLASSIFICATION MODELS

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

# 1. Define categorical and numerical features
categorical_features = [
    "type",
    "country",
    "listed_in",
    "director"
]

numerical_features = [
    "release_year"
]

# 2. Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 3. Create preprocessing transformer
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical",
         OneHotEncoder(handle_unknown="ignore"),
         categorical_features),

        ("numerical",
         "passthrough",
         numerical_features)
    ]
)

# 4. Create classification models
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
}

# 5. Train each model using a pipeline
trained_models = {}

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    pipeline.fit(X_train, y_train)

    trained_models[name] = pipeline

    print(name, "trained successfully.")

# 6. Display dataset shapes
print("\nTraining Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)

Decision Tree trained successfully.
Random Forest trained successfully.

Training Data Shape: (7032, 5)
Testing Data Shape: (1758, 5)


OPTIMIZE MODEL PERFORMANCE

In [9]:
from sklearn.model_selection import RandomizedSearchCV

# 1. Define parameters for Decision Tree
dt_params = {
    "classifier__max_depth": [None, 10, 20, 30],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4]
}

# 2. Define parameters for Random Forest
rf_params = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 20, 30],
    "classifier__min_samples_split": [2, 5],
    "classifier__min_samples_leaf": [1, 2]
}

# 3. Tune Decision Tree
dt_search = RandomizedSearchCV(
    estimator=trained_models["Decision Tree"],
    param_distributions=dt_params,
    n_iter=5,
    cv=3,
    scoring="f1_macro",
    random_state=42,
    n_jobs=-1
)

dt_search.fit(X_train, y_train)

# 4. Tune Random Forest
rf_search = RandomizedSearchCV(
    estimator=trained_models["Random Forest"],
    param_distributions=rf_params,
    n_iter=5,
    cv=3,
    scoring="f1_macro",
    random_state=42,
    n_jobs=-1
)

rf_search.fit(X_train, y_train)

# 5. Store optimized models
best_models = {
    "Decision Tree": dt_search.best_estimator_,
    "Random Forest": rf_search.best_estimator_
}

# 6. Display best parameters and scores
print("Decision Tree Best Parameters:")
print(dt_search.best_params_)
print("Best Cross-Validation F1 Score:", round(dt_search.best_score_, 4))

print("\nRandom Forest Best Parameters:")
print(rf_search.best_params_)
print("Best Cross-Validation F1 Score:", round(rf_search.best_score_, 4))

/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(


Decision Tree Best Parameters:
{'classifier__min_samples_split': 10, 'classifier__min_samples_leaf': 4, 'classifier__max_depth': 30}
Best Cross-Validation F1 Score: 0.289

Random Forest Best Parameters:
{'classifier__n_estimators': 100, 'classifier__min_samples_split': 2, 'classifier__min_samples_leaf': 1, 'classifier__max_depth': None}
Best Cross-Validation F1 Score: 0.3069


EVALUATE PREDICTION ACCURACY

In [10]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

results = []

# 1. Evaluate each optimized model
for name, model in best_models.items():

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Macro F1 Score": macro_f1
    })

    print("\nModel:", name)
    print("Accuracy:", round(accuracy, 4))
    print("Macro F1 Score:", round(macro_f1, 4))

    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        zero_division=0
    ))

# 2. Compare model performance
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
)

print("\nMODEL PERFORMANCE COMPARISON:")
display(results_df)

# 3. Identify the best model
best_model = results_df.iloc[0]

print("\nBest Model:", best_model["Model"])
print("Best Accuracy:", round(best_model["Accuracy"] * 100, 2), "%")


Model: Decision Tree
Accuracy: 0.4983
Macro F1 Score: 0.2743

Classification Report:
              precision    recall  f1-score   support

           G       0.36      0.50      0.42         8
       NC-17       0.00      0.00      0.00         1
          NR       0.00      0.00      0.00        16
          PG       0.43      0.39      0.41        57
       PG-13       0.33      0.23      0.27        98
           R       0.43      0.44      0.43       160
       TV-14       0.50      0.45      0.47       431
        TV-G       0.25      0.05      0.08        44
       TV-MA       0.55      0.75      0.63       641
       TV-PG       0.32      0.13      0.19       172
        TV-Y       0.54      0.51      0.53        61
       TV-Y7       0.49      0.34      0.40        67
    TV-Y7-FV       0.00      0.00      0.00         1
          UR       0.00      0.00      0.00         1

    accuracy                           0.50      1758
   macro avg       0.30      0.27      0.27     

,Model,Accuracy,Macro F1 Score
1,Random Forest,0.521615,0.323032
0,Decision Tree,0.498294,0.274289



Best Model: Random Forest
Best Accuracy: 52.16 %
